# Zenith on Colab

1. Runtime > Change runtime type > GPU.
2. Run the setup cell. It pulls the latest code from GitHub and installs packages.
3. Set the run name and learning rate in the run cell, then run it.

Data is generated on first use and saved to Google Drive, under `MyDrive/zenith/`. Results go to `MyDrive/zenith/results/<name>/`. Re-running a finished name only re-runs the evaluation.

In [ ]:
from google.colab import drive, userdata
import base64, subprocess, os
drive.mount('/content/drive')
DRIVE = '/content/drive/MyDrive/zenith'
DATA = f'{DRIVE}/zenith_data'
REPO = '/content/zenith'
REMOTE = 'https://github.com/rbf22/zenith.git'
try:
    token = userdata.get('GITHUB_TOKEN')
except Exception:
    token = None
header = ['-c', f'http.extraheader=Authorization: basic {base64.b64encode(f"x-access-token:{token}".encode()).decode()}'] if token else []
def git(*args, cwd=None):
    subprocess.run(['git', *header, *args], cwd=cwd, check=True)
if os.path.exists(f'{REPO}/.git'):
    git('fetch', '-q', 'origin', cwd=REPO)
    git('reset', '-q', '--hard', 'origin/main', cwd=REPO)
else:
    try:
        git('clone', '-q', REMOTE, REPO)
    except subprocess.CalledProcessError:
        raise SystemExit('Clone failed. The repo is private: add a GITHUB_TOKEN secret (read access to rbf22/zenith), or make the repo public.')
print(subprocess.run(['git', 'log', '--oneline', '-1'], cwd=REPO, capture_output=True, text=True).stdout)

In [ ]:
!pip install -q transformers datasets tokenizers pytest pytest-cov

In [ ]:
%cd /content/zenith
RUN_NAME = 'accum8_lr2e-4'   # any new name for a new run
LR = '2e-4'
STEPS = 38
GRAD_ACCUM = 8
!python scripts/run_experiment.py --drive /content/drive/MyDrive/zenith --name {RUN_NAME} --lr {LR} --steps {STEPS} --grad_accum {GRAD_ACCUM}